<a href="https://colab.research.google.com/github/EmperorBlackMD/BME-6720/blob/main/BME6720_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Week 1: MSD Spleen Dataset Initial Check
import os
import json
import pandas as pd
import tarfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Setting dataset path
tar_path = Path('/content/drive/MyDrive/BME6720_Project/Task09_Spleen.tar')
DATASET_DIR = Path('/content/drive/MyDrive/BME6720_Project/')

with tarfile.open(tar_path, 'r') as tar:
    tar.extractall(path=DATASET_DIR)
print('Extraction complete')

for root, dirs, files in os.walk(DATASET_DIR):
    print(root)


/tmp/ipykernel_2292/1469869667.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=DATASET_DIR)


Extraction complete
/content/drive/MyDrive/BME6720_Project
/content/drive/MyDrive/BME6720_Project/Task09_Spleen
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTs


In [ ]:
# Finalizing dataset path

DATASET_DIR1 = Path('/content/drive/MyDrive/BME6720_Project/Task09_Spleen')

imagesTr_dir = DATASET_DIR1 / 'imagesTr'
labelsTr_dir = DATASET_DIR1 / 'labelsTr'
imagesTs_dir = DATASET_DIR1 / 'imagesTs'
dataset_json_path = DATASET_DIR1 / 'dataset.json'

expected_paths = {
    'Dataset root': DATASET_DIR1,
    'Training images folder': imagesTr_dir,
    'Training labels folder': labelsTr_dir,
    'Test images folder': imagesTs_dir,
    'Dataset JSON': dataset_json_path
}

print('Dataset structure check')
for name, path in expected_paths.items():
    print(f'{name}: {'FOUND' if path.exists() else 'NOT FOUND'} --> {path}')

Dataset structure check
Dataset root: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen
Training images folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr
Training labels folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr
Test images folder: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTs
Dataset JSON: FOUND --> /content/drive/MyDrive/BME6720_Project/Task09_Spleen/dataset.json


In [ ]:
# List image and label files
train_images = sorted(list(imagesTr_dir.glob('*.nii.gz')))
train_labels = sorted(list(labelsTr_dir.glob('*.nii.gz')))
test_images = sorted(list(imagesTs_dir.glob('*.nii.gz')))

print(f'Number of Training images: {len(train_images)}')
print(f'Number of Training labels: {len(train_labels)}')
print(f'Number of Test images: {len(test_images)}')

print('\nFirst 5 Training images:')
for image in train_images[:5]:
    print(image)

print('\nFirst 5 Training labels:')
for label in train_labels[:5]:
    print(label)


Number of Training images: 82
Number of Training labels: 82
Number of Test images: 40

First 5 Training images:
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_10.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_12.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_13.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_14.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/imagesTr/._spleen_16.nii.gz

First 5 Training labels:
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_10.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_12.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_13.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_14.nii.gz
/content/drive/MyDrive/BME6720_Project/Task09_Spleen/labelsTr/._spleen_16.nii.gz


In [ ]:
# Read dataset.json metadata
with open(dataset_json_path, 'r') as f:
    dataset_json = json.load(f)

print('Dataset metadata:')
print(json.dumps(dataset_json, indent=4))


Dataset metadata:
{
    "name": "Spleen",
    "description": "Spleen Segmentation",
    "reference": "Memorial Sloan Kettering Cancer Center",
    "licence": "CC-BY-SA 4.0",
    "release": "1.0 06/08/2018",
    "tensorImageSize": "3D",
    "modality": {
        "0": "CT"
    },
    "labels": {
        "0": "background",
        "1": "spleen"
    },
    "numTraining": 41,
    "numTest": 20,
    "training": [
        {
            "image": "./imagesTr/spleen_19.nii.gz",
            "label": "./labelsTr/spleen_19.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_31.nii.gz",
            "label": "./labelsTr/spleen_31.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_52.nii.gz",
            "label": "./labelsTr/spleen_52.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_40.nii.gz",
            "label": "./labelsTr/spleen_40.nii.gz"
        },
        {
            "image": "./imagesTr/spleen_3.nii.gz",
            "label": "./labelsTr

In [ ]:
# Creating case-level manifest

def get_case_id(filename):
    """
    Extracts case ID from MSD-style filenames.
    Example: spleen_10.nii.gz -> spleen_10
    """
    return filename.replace('.nii.gz', '')

image_case_ids = [get_case_id(image.name) for image in train_images]
label_case_ids = [get_case_id(label.name) for label in train_labels]
manifest = []

for image_file in train_images:
    case_id = get_case_id(image_file.name)
    expected_label = labelsTr_dir / image_file.name

    manifest.append({
        'case_id': case_id,
        'image_file': str(image_file),
        'label_file': str(expected_label),
        'label_exists': expected_label.exists()
    })
manifest_df = pd.DataFrame(manifest)
manifest_df.head()

,case_id,image_file,label_file,label_exists
0,._spleen_10,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
1,._spleen_12,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
2,._spleen_13,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
3,._spleen_14,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
4,._spleen_16,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True


In [ ]:
from IPython.core.interactiveshell import dis
# Check for missing labels or unmatched files
missing_labels = manifest_df[manifest_df['label_exists'] == False]

print(f'Total image-label pairs expected: {len(manifest_df)}')
print(f'Total missing label files: {len(missing_labels)}')

if len(missing_labels) > 0:
    display(missing_labels)
else:
    print('All training images have corresponding labels')


Total image-label pairs expected: 82
Total missing label files: 0
All training images have corresponding labels


In [ ]:
# Saving manifest for future weekly updates
output_manifest_path = DATASET_DIR1 / 'week1dataset_manifest.csv'
manifest_df.to_csv(output_manifest_path, index=False)
print(f'Manifest saved to {output_manifest_path}')

Manifest saved to /content/drive/MyDrive/BME6720_Project/Task09_Spleen/week1dataset_manifest.csv


# Week 3

In [1]:
!pip install nibabel pandas numpy tqdm -q

In [2]:
from pathlib import Path
import nibabel as nib
import numpy as np
import pandas as pd
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Uploading the dataset path
DATASET_DIR = Path('/content/drive/MyDrive/BME6720_Project/Task09_Spleen')
manifest_path = DATASET_DIR / 'week1dataset_manifest.csv'
print('Dataset diretoy exists:', DATASET_DIR.exists())
print('Manifest exists:', manifest_path.exists())

Dataset diretoy exists: True
Manifest exists: True


In [4]:
# Reloading saved manifest
manifest_df = pd.read_csv(manifest_path)
print('Manifest shape:', manifest_df.shape)
manifest_df.head()

Manifest shape: (82, 4)


,case_id,image_file,label_file,label_exists
0,._spleen_10,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
1,._spleen_12,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
2,._spleen_13,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
3,._spleen_14,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True
4,._spleen_16,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True


In [10]:
#Manifest cleanup
import os
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.startswith('._'):
            os.remove(os.path.join(root, file))
print('Removed unnecessary metadata files')

Removed unnecessary metadata files


In [11]:
len(manifest_df)

82

In [13]:
# Confirming if files are still all accessible and cleaning the data
manifest_df['image_exists'] = manifest_df['image_file'].apply(lambda x: Path(x).exists())
manifest_df['label_exists'] = manifest_df['label_file'].apply(lambda x: Path(x).exists())

clean_manifest_df = manifest_df[
    (manifest_df['image_exists'] == True) &
    (manifest_df['label_exists'] == True)
].copy()

print('Original manifest rows:', len(manifest_df))
print('Cleaned manifest rows:', len(clean_manifest_df))

clean_manifest_df.head()

Original manifest rows: 82
Cleaned manifest rows: 41


,case_id,image_file,label_file,label_exists,image_exists
41,spleen_10,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
42,spleen_12,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
43,spleen_13,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
44,spleen_14,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
45,spleen_16,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True


In [14]:
# Saving cleaned manifest
clean_manifest_path = DATASET_DIR / 'week3dataset_manifest_clean.csv'
clean_manifest_df.to_csv(clean_manifest_path, index=False)
print('Cleaned manifest saved to:', clean_manifest_path)

Cleaned manifest saved to: /content/drive/MyDrive/BME6720_Project/Task09_Spleen/week3dataset_manifest_clean.csv


In [15]:
# Reloading cleaned manifest
manifest_df = pd.read_csv(clean_manifest_path)
print('Manifest shape:', manifest_df.shape)
manifest_df.head()

Manifest shape: (41, 5)


,case_id,image_file,label_file,label_exists,image_exists
0,spleen_10,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
1,spleen_12,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
2,spleen_13,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
3,spleen_14,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True
4,spleen_16,/content/drive/MyDrive/BME6720_Project/Task09_...,/content/drive/MyDrive/BME6720_Project/Task09_...,True,True


In [16]:
# Inspecting a sigle case
sample_row = manifest_df.iloc[0]
image_path = sample_row['image_file']
label_path = sample_row['label_file']

image_nii = nib.load(image_path)
label_nii = nib.load(label_path)

image_data = image_nii.get_fdata()
label_data = label_nii.get_fdata()

print('Case ID:', sample_row['case_id'])
print('Image shape:', image_data.shape)
print('Label shape:', label_data.shape)
print('Voxel spacing:', image_nii.header.get_zooms()[:3])
print('Unique label values:', np.unique(label_data))

Case ID: spleen_10
Image shape: (512, 512, 55)
Label shape: (512, 512, 55)
Voxel spacing: (np.float32(0.976562), np.float32(0.976562), np.float32(5.0))
Unique label values: [0. 1.]


In [19]:
# Extracting image shape and spacing for all cases
summary_rows = []
for _, row in tqdm(manifest_df.iterrows(), total=len(manifest_df)):
    case_id = ['case_id']
    image_path = Path(row['image_file'])
    label_path = Path(row['label_file'])

    image_nii = nib.load(image_path)
    label_nii = nib.load(label_path)

    image_shape = image_nii.shape
    label_shape = label_nii.shape
    image_spacing = image_nii.header.get_zooms()[:3]

    summary_rows.append({
        'case_id': case_id,
        'image_dim_x': image_shape[0],
        'image_dim_y': image_shape[1],
        'image_dim_z': image_shape[2],
        'label_dim_x': label_shape[0],
        'label_dim_y': label_shape[1],
        'label_dim_z': label_shape[2],
        'image_spacing_x_mm': image_spacing[0],
        'image_spacing_y_mm': image_spacing[1],
        'image_spacing_z_mm': image_spacing[2],

        'image_label_shape_match': image_shape == label_shape
    })
shape_spacing_df = pd.DataFrame(summary_rows)
shape_spacing_df.head()


100%|██████████| 41/41 [00:57<00:00,  1.39s/it]


,case_id,image_dim_x,image_dim_y,image_dim_z,label_dim_x,label_dim_y,label_dim_z,image_spacing_x_mm,image_spacing_y_mm,image_spacing_z_mm,image_label_shape_match
0,[case_id],512,512,55,512,512,55,0.976562,0.976562,5.0,True
1,[case_id],512,512,168,512,512,168,0.753906,0.753906,1.5,True
2,[case_id],512,512,77,512,512,77,0.742188,0.742188,2.5,True
3,[case_id],512,512,54,512,512,54,0.851562,0.851562,5.0,True
4,[case_id],512,512,61,512,512,61,0.792969,0.792969,8.0,True


In [20]:
# Adding total voxel count and physical scan volume
shape_spacing_df['total_voxels'] = (shape_spacing_df['image_dim_x'] * shape_spacing_df['image_dim_y'] * shape_spacing_df['image_dim_z'])
shape_spacing_df['voxel_volume_mm3'] = (
    shape_spacing_df['image_spacing_x_mm'] *
    shape_spacing_df['image_spacing_y_mm'] *
    shape_spacing_df['image_spacing_z_mm']
)
shape_spacing_df['physical_scan_volume_mm3'] = (
    shape_spacing_df['total_voxels'] * shape_spacing_df['voxel_volume_mm3']
)
shape_spacing_df['scan_volume_liters'] = (shape_spacing_df['physical_scan_volume_mm3'] / 1_000_000)
shape_spacing_df.head()

,case_id,image_dim_x,image_dim_y,image_dim_z,label_dim_x,label_dim_y,label_dim_z,image_spacing_x_mm,image_spacing_y_mm,image_spacing_z_mm,image_label_shape_match,total_voxels,voxel_volume_mm3,physical_scan_volume_mm3,scan_volume_liters
0,[case_id],512,512,55,512,512,55,0.976562,0.976562,5.0,True,14417920,4.768367,6.874993e+07,68.749931
1,[case_id],512,512,168,512,512,168,0.753906,0.753906,1.5,True,44040192,0.852561,3.754697e+07,37.546968
2,[case_id],512,512,77,512,512,77,0.742188,0.742188,2.5,True,20185088,1.377108,2.779704e+07,27.797036
3,[case_id],512,512,54,512,512,54,0.851562,0.851562,5.0,True,14155776,3.625789,5.132586e+07,51.325859
4,[case_id],512,512,61,512,512,61,0.792969,0.792969,8.0,True,15990784,5.030398,8.044001e+07,80.440014


In [21]:
# Generating summary statistics
shape_spacing_df.describe()

,image_dim_x,image_dim_y,image_dim_z,label_dim_x,label_dim_y,label_dim_z,image_spacing_x_mm,image_spacing_y_mm,image_spacing_z_mm,total_voxels,voxel_volume_mm3,physical_scan_volume_mm3,scan_volume_liters
count,41.0,41.0,41.000000,41.0,41.0,41.000000,41.000000,41.000000,41.000000,4.100000e+01,41.000000,4.100000e+01,41.000000
mean,512.0,512.0,89.024390,512.0,512.0,89.024390,0.812405,0.812405,4.368292,2.333721e+07,2.876291,6.103546e+07,61.035463
std,0.0,0.0,36.752203,0.0,0.0,36.752203,0.097809,0.097809,1.552808,9.634369e+06,1.142730,2.502018e+07,25.020176
min,512.0,512.0,31.000000,512.0,512.0,31.000000,0.613281,0.613281,1.500000,8.126464e+06,0.852561,2.341653e+07,23.416532
25%,512.0,512.0,60.000000,512.0,512.0,60.000000,0.742188,0.742188,4.000000,1.572864e+07,2.096377,4.021944e+07,40.219442
50%,512.0,512.0,90.000000,512.0,512.0,90.000000,0.792969,0.792969,5.000000,2.359296e+07,2.827167,5.757799e+07,57.577994
75%,512.0,512.0,103.000000,512.0,512.0,103.000000,0.902344,0.902344,5.000000,2.700083e+07,3.693607,7.919217e+07,79.192175
max,512.0,512.0,168.000000,512.0,512.0,168.000000,0.976562,0.976562,8.000000,4.404019e+07,5.030398,1.109909e+08,110.990932


In [22]:
shape_spacing_df[
    [
        'image_dim_x', 'image_dim_y', 'image_dim_z',
        'image_spacing_x_mm', 'image_spacing_y_mm', 'image_spacing_z_mm',
        'total_voxels', 'voxel_volume_mm3', 'physical_scan_volume_mm3'
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
image_dim_x,41.0,5.120000e+02,0.000000e+00,5.120000e+02,5.120000e+02,5.120000e+02,5.120000e+02,5.120000e+02
image_dim_y,41.0,5.120000e+02,0.000000e+00,5.120000e+02,5.120000e+02,5.120000e+02,5.120000e+02,5.120000e+02
image_dim_z,41.0,8.902439e+01,3.675220e+01,3.100000e+01,6.000000e+01,9.000000e+01,1.030000e+02,1.680000e+02
image_spacing_x_mm,41.0,8.124048e-01,9.780948e-02,6.132810e-01,7.421880e-01,7.929690e-01,9.023440e-01,9.765620e-01
image_spacing_y_mm,41.0,8.124048e-01,9.780948e-02,6.132810e-01,7.421880e-01,7.929690e-01,9.023440e-01,9.765620e-01
image_spacing_z_mm,41.0,4.368292e+00,1.552808e+00,1.500000e+00,4.000000e+00,5.000000e+00,5.000000e+00,8.000000e+00
total_voxels,41.0,2.333721e+07,9.634369e+06,8.126464e+06,1.572864e+07,2.359296e+07,2.700083e+07,4.404019e+07
voxel_volume_mm3,41.0,2.876291e+00,1.142730e+00,8.525614e-01,2.096377e+00,2.827167e+00,3.693607e+00,5.030398e+00
physical_scan_volume_mm3,41.0,6.103546e+07,2.502018e+07,2.341653e+07,4.021944e+07,5.757799e+07,7.919217e+07,1.109909e+08


In [23]:
# Checking for problems
shape_mismatch_cases = shape_spacing_df[shape_spacing_df['image_label_shape_match'] == False]
print('Cases with mismatched image and label shapes:', len(shape_mismatch_cases))
shape_mismatch_cases

Cases with mismatched image and label shapes: 0


,case_id,image_dim_x,image_dim_y,image_dim_z,label_dim_x,label_dim_y,label_dim_z,image_spacing_x_mm,image_spacing_y_mm,image_spacing_z_mm,image_label_shape_match,total_voxels,voxel_volume_mm3,physical_scan_volume_mm3,scan_volume_liters


In [24]:
# Saving summary table
summary_table_path = DATASET_DIR / 'week3dataset_summary_table.csv'
shape_spacing_df.to_csv(summary_table_path, index=False)
print('Summary table saved to:', summary_table_path)

Summary table saved to: /content/drive/MyDrive/BME6720_Project/Task09_Spleen/week3dataset_summary_table.csv


In [25]:
# Display final table
shape_spacing_df.head(10)

,case_id,image_dim_x,image_dim_y,image_dim_z,label_dim_x,label_dim_y,label_dim_z,image_spacing_x_mm,image_spacing_y_mm,image_spacing_z_mm,image_label_shape_match,total_voxels,voxel_volume_mm3,physical_scan_volume_mm3,scan_volume_liters
0,[case_id],512,512,55,512,512,55,0.976562,0.976562,5.0,True,14417920,4.768367,6.874993e+07,68.749931
1,[case_id],512,512,168,512,512,168,0.753906,0.753906,1.5,True,44040192,0.852561,3.754697e+07,37.546968
2,[case_id],512,512,77,512,512,77,0.742188,0.742188,2.5,True,20185088,1.377108,2.779704e+07,27.797036
3,[case_id],512,512,54,512,512,54,0.851562,0.851562,5.0,True,14155776,3.625789,5.132586e+07,51.325859
4,[case_id],512,512,61,512,512,61,0.792969,0.792969,8.0,True,15990784,5.030398,8.044001e+07,80.440014
5,[case_id],512,512,95,512,512,95,0.613281,0.613281,2.5,True,24903680,0.940284,2.341653e+07,23.416532
6,[case_id],512,512,164,512,512,164,0.966797,0.966797,1.5,True,42991616,1.402045,6.027617e+07,60.276165
7,[case_id],512,512,51,512,512,51,0.796875,0.796875,5.0,True,13369344,3.175049,4.244832e+07,42.448320
8,[case_id],512,512,90,512,512,90,0.794922,0.794922,5.0,True,23592960,3.159505,7.454207e+07,74.542073
9,[case_id],512,512,168,512,512,168,0.933594,0.933594,1.5,True,44040192,1.307397,5.757799e+07,57.577994
